# Azure Deny Assignment - Protect a Route Table (UDR)

This notebook demonstrates an **Azure deny assignment** that prevents modification and deletion of an Azure **Route Table (User Defined Route / UDR)**, based on the requirement in [NOTES.md](NOTES.md) and the [New-AzDenyAssignment](https://learn.microsoft.com/powershell/module/az.resources/new-azdenyassignment) reference.

The infrastructure is described as **Bicep** and deployed with the **Azure CLI** (`az`). Every command in this notebook is plain **bash + az CLI** (no PowerShell, no Python SDK).

## What is a deny assignment?

A **deny assignment** attaches a set of *denied actions* to a principal at a scope. Unlike Azure Policy or RBAC role assignments, **deny assignments override every role assignment - including `Owner`**. If an action is denied, even a subscription Owner cannot perform it.

Azure now supports **user-assigned deny assignments** that you can create yourself (previously only Azure Blueprints / Deployment Stacks could create them). They are exposed through the ARM resource type `Microsoft.Authorization/denyAssignments@2024-07-01-preview`, which means they can be declared in **Bicep**.

> Only `write`, `delete` and `action` operations can be denied. Read actions and data actions are not supported.

## How the demonstration works (with and without)

```
  routetable.bicep                         denyassignment.bicep
  +-------------------+                     +-----------------------------+
  | Route Table (UDR) |  <---- protects ----| denyAssignments@2024-07-01  |
  |  forceTunneling   |                     |  actions: write/delete on   |
  |  0.0.0.0/0 -> NVA |                     |  routeTables + routes       |
  +-------------------+                     |  principals: All Principals |
                                            |  EXCLUDED: admin (ga1)      |
                                            +-----------------------------+
```

The **actor** that performs the route changes is the privileged user `e1328847-0389-455c-a911-1f7bd1d1deee` (Owner on the subscription). Their commands run in a separate Azure CLI context so they truly execute *as that user*, not as your admin login.

> **Key constraint:** when a deny assignment targets **All Principals**, Azure requires **at least one excluded principal**. We exclude the admin (`ga1`) - this is mandatory *and* doubles as the built-in break-glass path: the admin can always fix the UDR while everyone else is blocked.

1. Deploy the route table (UDR) with Bicep (admin).
2. Grant the actor Owner on the subscription and sign in as them (Step 1a).
3. **WITHOUT** the deny assignment -> the actor modifying / deleting the UDR **succeeds**.
4. Deploy the deny assignment with Bicep (admin), excluding the admin principal.
5. **WITH** the deny assignment -> the same actor's modify / delete attempts are **blocked** (even as Owner), while the excluded admin can still change the UDR.
6. Switch the effect to **audit** to observe instead of block (Step 5).
7. Show that a user-assigned deny assignment **cannot self-protect** against deletion - a targeted Owner can delete it to bypass it (Step 6).
8. Clean up (remove the deny assignment first, then the resource group).

## Prerequisites

- **Azure CLI** installed and signed in (`az login`).
- A **Bash** Jupyter kernel (install with `pip install bash_kernel && python -m bash_kernel.install`). Variables set in one cell persist to the next.
- Permissions to create deny assignments at the target scope: **Owner** or a role with `Microsoft.Authorization/denyAssignments/write` (for example *Role Based Access Control Administrator*).
- `jq` is helpful but not required.

> **Note on the subscription ID.** The value supplied for this exercise (`7777febc-d5d8-44da-b990-643c836`) is **not a complete GUID** (a subscription ID has the form `xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx`). Replace `SUBSCRIPTION_ID` below with your full subscription ID before running. The `az account set` cell will fail fast if the ID is invalid.

In [24]:
# --- Configuration -------------------------------------------------------
# Set SUBSCRIPTION_ID to your full subscription GUID. If it is invalid or
# inaccessible, the cell falls back to your currently signed-in default.
export SUBSCRIPTION_ID="7777febc-d5d8-44da-b990-643c8369f1e1"
export LOCATION="germanywestcentral"
export PREFIX="cptdazdeny"
export RG="${PREFIX}"
export ROUTE_TABLE="${PREFIX}-rt"

# Helper (defined here so EVERY following cell can call it) that prints which
# Azure AD identity / context the current cell runs as.
# $1 = role label (e.g. "ACTOR (test1)" or "ADMIN (ga1)"); $2 = optional AZURE_CONFIG_DIR.
print_ctx () {
  local who
  if [ -n "${2:-}" ]; then
    who=$(AZURE_CONFIG_DIR="$2" az ad signed-in-user show --query "join(' | ', [userPrincipalName, id])" -o tsv 2>/dev/null)
  else
    who=$(az ad signed-in-user show --query "join(' | ', [userPrincipalName, id])" -o tsv 2>/dev/null)
  fi
  echo "### RUNNING AS: ${1} | live signed-in user: ${who:-<unknown>} ###"
}

# Try to select the requested subscription; if it is invalid or inaccessible,
# fall back to the subscription of the current login.
if ! az account set --subscription "${SUBSCRIPTION_ID}" 2>/dev/null; then
  echo "WARNING: subscription '${SUBSCRIPTION_ID}' is invalid or inaccessible."
  export SUBSCRIPTION_ID=$(az account show --query id -o tsv)
  echo "         Falling back to the current default subscription: ${SUBSCRIPTION_ID}"
  az account set --subscription "${SUBSCRIPTION_ID}"
fi

# CONTEXT: this cell runs as the ADMIN (your default az login).
print_ctx "ADMIN (ga1)"

echo "Subscription : ${SUBSCRIPTION_ID}"
echo "Location     : ${LOCATION}"
echo "Resource grp : ${RG}"
echo "Route table  : ${ROUTE_TABLE}"

az account show --query "{name:name, subscriptionId:id, tenantId:tenantId}" -o table

### RUNNING AS: ADMIN (ga1) | live signed-in user: ga1@cptdx.net | c1a139f7-6a1c-4c45-ae43-33c9b758e9c3 ###
Subscription : 7777febc-d5d8-44da-b990-643c8369f1e1
Location     : germanywestcentral
Resource grp : cptdazdeny
Route table  : cptdazdeny-rt
Name          SubscriptionId                        TenantId
------------  ------------------------------------  ------------------------------------
sub-cptdx-04  7777febc-d5d8-44da-b990-643c8369f1e1  ade68923-b72b-4190-8508-a19a58692001


In [25]:
# CONTEXT: this cell runs as the ADMIN (your default az login).
print_ctx "ADMIN (ga1)"

# Who am I? (object ID is useful if you later want to exclude a break-glass admin)
export MY_OBJECT_ID=$(az ad signed-in-user show --query id -o tsv)
export MY_UPN=$(az ad signed-in-user show --query userPrincipalName -o tsv)
echo "Signed-in user : ${MY_UPN}"
echo "Object ID      : ${MY_OBJECT_ID}"

### RUNNING AS: ADMIN (ga1) | live signed-in user: ga1@cptdx.net | c1a139f7-6a1c-4c45-ae43-33c9b758e9c3 ###
Signed-in user : ga1@cptdx.net
Object ID      : c1a139f7-6a1c-4c45-ae43-33c9b758e9c3


## Step 1a - Set up the privileged user as the actor

The scenario: user `e1328847-0389-455c-a911-1f7bd1d1deee` (the **actor**) is granted **Owner on the subscription** and performs all the route changes in Steps 2 and 4. Your current admin login only sets up the infrastructure and the deny assignment.

So the actor's commands really run *as that user* - and not as the admin - the notebook keeps a **separate Azure CLI context** in `AZURE_CONFIG_DIR=${TARGET_CFG}`. The admin keeps its own default login untouched. The next two cells grant the role and sign the actor in.

In [26]:
# CONTEXT: this cell runs as the ADMIN - it grants the ACTOR (test1) Owner on the subscription.
print_ctx "ADMIN (ga1)"

# The ACTOR for this demo: gets Owner on the whole subscription and performs all
# the route changes. The admin (your default login) only sets up infra + deny.
export TARGET_USER_OBJECT_ID="e1328847-0389-455c-a911-1f7bd1d1deee"
export TARGET_CFG=/tmp/az-target-user

echo ">> Granting Owner on the SUBSCRIPTION to the actor (${TARGET_USER_OBJECT_ID})..."
az role assignment create \
  --assignee-object-id "${TARGET_USER_OBJECT_ID}" \
  --assignee-principal-type User \
  --role "Owner" \
  --scope "/subscriptions/${SUBSCRIPTION_ID}" \
  --query "{role:roleDefinitionName, principalId:principalId, scope:scope}" -o table

### RUNNING AS: ADMIN (ga1) | live signed-in user: ga1@cptdx.net | c1a139f7-6a1c-4c45-ae43-33c9b758e9c3 ###
>> Granting Owner on the SUBSCRIPTION to the actor (e1328847-0389-455c-a911-1f7bd1d1deee)...
Role    PrincipalId                           Scope
------  ------------------------------------  ---------------------------------------------------
Owner   e1328847-0389-455c-a911-1f7bd1d1deee  /subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1


In [27]:
# CONTEXT: the OUTER shell is the ADMIN; this cell signs the ACTOR (test1) into a
# separate, isolated CLI context (AZURE_CONFIG_DIR=${TARGET_CFG}) so the actor's
# commands never touch your admin login.
echo "### SETUP: ADMIN shell signing the ACTOR (test1) into AZURE_CONFIG_DIR=${TARGET_CFG} ###"
mkdir -p "${TARGET_CFG}"

# Disable the interactive "Tenant and subscription selection" prompt that newer
# az versions show after device-code login - it hangs a non-interactive kernel.
AZURE_CONFIG_DIR="${TARGET_CFG}" az config set core.login_experience_v2=off --only-show-errors

echo ">> Sign in as the actor (${TARGET_USER_OBJECT_ID})."
echo "   Complete the device-code prompt below with THAT user's credentials."
AZURE_CONFIG_DIR="${TARGET_CFG}" az login --use-device-code --only-show-errors -o none
AZURE_CONFIG_DIR="${TARGET_CFG}" az account set --subscription "${SUBSCRIPTION_ID}"

echo ""
echo ">> Confirming the actor context identity..."
print_ctx "ACTOR (test1)" "${TARGET_CFG}"
TARGET_CTX_ID=$(AZURE_CONFIG_DIR="${TARGET_CFG}" az ad signed-in-user show --query id -o tsv)
if [ "${TARGET_CTX_ID}" = "${TARGET_USER_OBJECT_ID}" ]; then
  echo ">> OK - the actor context is the expected user; Steps 2 and 4 run as them."
else
  echo ">> WARNING - signed-in user (${TARGET_CTX_ID}) is not ${TARGET_USER_OBJECT_ID}."
  echo "   Re-run 'az login' as the correct user before continuing."
fi

### SETUP: ADMIN shell signing the ACTOR (test1) into AZURE_CONFIG_DIR=/tmp/az-target-user ###
>> Sign in as the actor (e1328847-0389-455c-a911-1f7bd1d1deee).
   Complete the device-code prompt below with THAT user's credentials.
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code LC73AV4GT to authenticate.

>> Confirming the actor context identity...
### RUNNING AS: ACTOR (test1) | live signed-in user: user1@cptdx.net | e1328847-0389-455c-a911-1f7bd1d1deee ###
>> OK - the actor context is the expected user; Steps 2 and 4 run as them.


## Step 1 - Deploy the Route Table (UDR) with Bicep

Create the resource group, then deploy [routetable.bicep](routetable.bicep), which creates a route table with a single force-tunneling route (`0.0.0.0/0` -> virtual appliance).

In [ ]:
# CONTEXT: this cell runs as the ADMIN (your default login) - it deploys the infrastructure.
print_ctx "ADMIN (ga1)"

az group create --name "${RG}" --location "${LOCATION}" -o table

az deployment group create \
  --resource-group "${RG}" \
  --name routetable \
  --template-file routetable.bicep \
  --parameters routeTableName="${ROUTE_TABLE}" \
  -o table

echo "--- Routes in ${ROUTE_TABLE} ---"
az network route-table route list \
  --resource-group "${RG}" \
  --route-table-name "${ROUTE_TABLE}" \
  --query "[].{name:name, prefix:addressPrefix, nextHopType:nextHopType, nextHopIp:nextHopIpAddress}" -o table

Location            Name
------------------  ----------
germanywestcentral  cptdazdeny
Name        State      Timestamp                         Mode         ResourceGroup
----------  ---------  --------------------------------  -----------  ---------------
routetable  Succeeded  2026-06-23T11:20:10.976004+00:00  Incremental  cptdazdeny
--- Routes in cptdazdeny-rt ---
Name            Prefix     NextHopType       NextHopIp
--------------  ---------  ----------------  -----------
forceTunneling  0.0.0.0/0  VirtualAppliance  10.0.0.4


## Step 2 - WITHOUT the deny assignment: the actor can modify the UDR

Before applying the deny assignment, the **actor** (`e1328847-0389-455c-a911-1f7bd1d1deee`, Owner on the subscription) can freely modify the UDR. Every command below runs in the actor's CLI context (`AZURE_CONFIG_DIR=${TARGET_CFG}`), so it executes **as that user**, not as the admin. It updates the existing route, adds a temporary route, and deletes it - all succeed.

In [28]:
# CONTEXT: runs as the ACTOR (test1) - every az call uses AZURE_CONFIG_DIR=${TARGET_CFG}.
print_ctx "ACTOR (test1)" "${TARGET_CFG}"
echo "(expected the actor object id: ${TARGET_USER_OBJECT_ID})"
echo ""

echo ">> Updating the next hop of the forceTunneling route..."
AZURE_CONFIG_DIR="${TARGET_CFG}" az network route-table route update \
  --resource-group "${RG}" \
  --route-table-name "${ROUTE_TABLE}" \
  --name forceTunneling \
  --next-hop-type VirtualAppliance \
  --next-hop-ip-address 10.0.0.5 \
  --query "{name:name, nextHopIp:nextHopIpAddress}" -o table

echo ">> Adding a temporary route..."
AZURE_CONFIG_DIR="${TARGET_CFG}" az network route-table route create \
  --resource-group "${RG}" \
  --route-table-name "${ROUTE_TABLE}" \
  --name tempRoute \
  --address-prefix 10.99.0.0/16 \
  --next-hop-type VnetLocal \
  --query "{name:name, prefix:addressPrefix}" -o table

echo ">> Deleting the temporary route..."
AZURE_CONFIG_DIR="${TARGET_CFG}" az network route-table route delete \
  --resource-group "${RG}" \
  --route-table-name "${ROUTE_TABLE}" \
  --name tempRoute

echo ""
echo "RESULT: the actor's modifications WITHOUT a deny assignment SUCCEEDED."

### RUNNING AS: ACTOR (test1) | live signed-in user: user1@cptdx.net | e1328847-0389-455c-a911-1f7bd1d1deee ###
(expected the actor object id: e1328847-0389-455c-a911-1f7bd1d1deee)

>> Updating the next hop of the forceTunneling route...
(DenyAssignmentAuthorizationFailed) The client 'user1@cptdx.net' with object id 'e1328847-0389-455c-a911-1f7bd1d1deee' has permission to perform action 'Microsoft.Network/routeTables/routes/write' on scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt/routes/forceTunneling'; however, the access is denied because of the deny assignment with name 'Deny modify and delete of protected route table (UDR)' and Id 'c703e3901de753a6874d4e268cbbbc85' at scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourcegroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt'.
Code: DenyAssignmentAuthorizationFailed
Message: The client 'user1@cptdx.net' with obj

## Step 3 - Apply the deny assignment with Bicep

Deploy [denyassignment.bicep](denyassignment.bicep). It creates a `Microsoft.Authorization/denyAssignments` resource scoped to the route table that denies `write` and `delete` on the route table and its routes for **All Principals** (the zero-GUID `SystemDefined` principal).

> **Azure requires at least one excluded principal.** A user-assigned deny assignment that targets **All Principals** (Everyone) must exclude **at least one** principal - otherwise the deployment fails with `UserAssignedDenyAssignmentPropertiesNotValid: exclude principals are not valid`. This is a platform guardrail that stops you from locking *literally everyone* (Microsoft included) out of a resource forever.

So this step excludes the deploying **admin** (`ga1`, `${MY_OBJECT_ID}`) as the break-glass identity. The result is exactly the target scenario:

- the actor (`test1`, `e1328847-0389-455c-a911-1f7bd1d1deee`) is covered by the deny assignment and can **no longer modify or delete** the route table (shown in Step 4), while
- the admin (`ga1`) stays **excluded** and keeps full control (and can clean up).

In [ ]:
# CONTEXT: this cell runs as the ADMIN (your default login) - it deploys the deny assignment.
print_ctx "ADMIN (ga1)"

# Azure requires a user-assigned deny assignment that targets All Principals to
# exclude AT LEAST ONE principal (otherwise: UserAssignedDenyAssignmentPropertiesNotValid).
# We exclude the deploying admin (ga1) as break-glass, so the actor (test1) is
# blocked while the admin keeps control and can clean up afterwards.
cat > /tmp/deny.params.json <<EOF
{
  "\$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
  "contentVersion": "1.0.0.0",
  "parameters": {
    "routeTableName": { "value": "${ROUTE_TABLE}" },
    "excludePrincipalIds": { "value": ["${MY_OBJECT_ID}"] }
  }
}
EOF

az deployment group create \
  --resource-group "${RG}" \
  --name denyassignment \
  --template-file denyassignment.bicep \
  --parameters @/tmp/deny.params.json \
  -o table

export DENY_ID=$(az deployment group show \
  --resource-group "${RG}" \
  --name denyassignment \
  --query properties.outputs.denyAssignmentId.value -o tsv)
echo "Deny assignment resource ID: ${DENY_ID}"

echo ""
echo "NOTE: Azure Resource Manager caches authorization data for up to 30 minutes."
echo "The deny usually takes effect within seconds, but if a block below does not"
echo "trigger immediately, wait a moment and re-run the WITH cell."

Name            State      Timestamp                         Mode         ResourceGroup
--------------  ---------  --------------------------------  -----------  ---------------
denyassignment  Succeeded  2026-06-23T11:50:02.135524+00:00  Incremental  cptdazdeny
Deny assignment resource ID: /subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt/providers/Microsoft.Authorization/denyAssignments/c703e390-1de7-53a6-874d-4e268cbbbc85

NOTE: Azure Resource Manager caches authorization data for up to 30 minutes.
The deny usually takes effect within seconds, but if a block below does not
trigger immediately, wait a moment and re-run the WITH cell.


In [ ]:
# CONTEXT: this cell runs as the ADMIN (your default login) - it reads the deny assignment back.
print_ctx "ADMIN (ga1)"

# Confirm the deny assignment exists. Read it back directly via its resource ID
# (${DENY_ID} comes from the deployment output in the previous cell) using the
# same api-version that created it - the older 2022-04-01 list API does not
# return user-assigned deny assignments and would print an empty list.
az rest --method get \
  --url "https://management.azure.com${DENY_ID}?api-version=2024-07-01-preview" \
  --query "{displayName:properties.denyAssignmentName, effect:properties.denyAssignmentEffect, actions:properties.permissions[0].actions, principals:properties.principals[].type, excluded:properties.excludePrincipals[].id}" -o jsonc

{
  "actions": [
    "Microsoft.Network/routeTables/write",
    "Microsoft.Network/routeTables/delete",
    "Microsoft.Network/routeTables/routes/write",
    "Microsoft.Network/routeTables/routes/delete"
  ],
  "displayName": "Deny modify and delete of protected route table (UDR)",
  "effect": null,
  "excluded": [
    "c1a139f7-6a1c-4c45-ae43-33c9b758e9c3"
  ],
  "principals": [
    "SystemDefined"
  ]
}


## Step 4 - WITH the deny assignment: the actor is blocked (admin stays in control)

The attempts are split into separate cells so you can inspect each result on its own. They run as the **actor** in their own CLI context (`AZURE_CONFIG_DIR=${TARGET_CFG}`) and use **raw ARM REST calls** (`az rest`) so the cells print the **full JSON deny response** - including the deny assignment id, the blocked principal and the exact denied action - not just the short CLI message.

- The first cell sets up shared helpers (`check_blocked`, `show_raw`) and confirms the actor identity.
- Attempts 1-3 (UPDATE route / ADD route / DELETE route table) are each expected to be **blocked**, even though the actor is an `Owner` on the subscription. Failures are captured so the notebook keeps running.
- The final cell switches back to the **admin** context and shows the other side of the coin: the excluded admin (`ga1`) can **still** modify the route - that is the break-glass exclusion from Step 3 in action.

In [29]:
# --- Step 4 setup: helpers shared by the attempt cells below ----------------
# The bash kernel keeps these function definitions for every following cell.

# Prints which Azure AD identity / context the current cell runs as.
# $1 = role label (e.g. "ACTOR (test1)" or "ADMIN (ga1)"); $2 = optional AZURE_CONFIG_DIR.
print_ctx () {
  local who
  if [ -n "${2:-}" ]; then
    who=$(AZURE_CONFIG_DIR="$2" az ad signed-in-user show --query "join(' | ', [userPrincipalName, id])" -o tsv 2>/dev/null)
  else
    who=$(az ad signed-in-user show --query "join(' | ', [userPrincipalName, id])" -o tsv 2>/dev/null)
  fi
  echo "### RUNNING AS: ${1} | live signed-in user: ${who:-<unknown>} ###"
}

check_blocked () {
  # $1 = path to a file holding the full command output (stdout + stderr).
  if grep -qiE "deny assignment|denyassignment|disallowed|RequestDisallowedByAzure|RBACAccessDenied|AuthorizationFailed|does not have authorization" "$1"; then
    echo ">> CLASSIFICATION: BLOCKED by a deny assignment (expected)."
  else
    echo ">> CLASSIFICATION: NOT blocked - ARM auth cache may still be warming up (retry within ~30 min)."
  fi
}

show_raw () {
  # $1 = label, $2 = path to the captured raw response.
  echo "----- RAW RESPONSE: ${1} (full ARM JSON, stdout + stderr) -----"
  cat "$2"
  echo ""
  echo "----- deny assignment detail parsed from the raw response -----"
  grep -ioE "deny assignment '[^']*'|object id '[^']*'|action '[^']*'|denied by[^\"]*" "$2" | sort -u || true
  echo "---------------------------------------------------------------"
}

ROUTES_API="2023-11-01"
RT_URL="https://management.azure.com/subscriptions/${SUBSCRIPTION_ID}/resourceGroups/${RG}/providers/Microsoft.Network/routeTables/${ROUTE_TABLE}"

# This setup cell and the following attempt cells run in the ACTOR context.
print_ctx "ACTOR (test1)" "${TARGET_CFG}"
echo "(expected the actor object id: ${TARGET_USER_OBJECT_ID})"

### RUNNING AS: ACTOR (test1) | live signed-in user: user1@cptdx.net | e1328847-0389-455c-a911-1f7bd1d1deee ###
(expected the actor object id: e1328847-0389-455c-a911-1f7bd1d1deee)


### Attempt 1 - Update the route (expected: BLOCKED)

**Expected output:** the raw ARM response is an HTTP 403 JSON error (`RequestDisallowedByAzure`, message "disallowed by deny assignment") for the `Microsoft.Network/routeTables/routes/write` action. The parsed detail lists the deny assignment id, the blocked principal (the actor) and the denied action, and the classification line prints `>> CLASSIFICATION: BLOCKED by a deny assignment (expected).` The route's next hop is **not** changed.

In [30]:
# CONTEXT: runs as the ACTOR (test1) - every az call uses AZURE_CONFIG_DIR=${TARGET_CFG}.
print_ctx "ACTOR (test1)" "${TARGET_CFG}"

echo "=== Attempt 1: UPDATE the forceTunneling route (as the actor, raw ARM PUT) ==="
# Raw ARM PUT so the full JSON deny response is shown (not just the short CLI message).
AZURE_CONFIG_DIR="${TARGET_CFG}" az rest --method put \
  --url "${RT_URL}/routes/forceTunneling?api-version=${ROUTES_API}" \
  --headers "Content-Type=application/json" \
  --body '{"properties":{"addressPrefix":"0.0.0.0/0","nextHopType":"VirtualAppliance","nextHopIpAddress":"10.0.0.9"}}' \
  > /tmp/deny_update.txt 2>&1 || true

show_raw "UPDATE route" /tmp/deny_update.txt
check_blocked /tmp/deny_update.txt

### RUNNING AS: ACTOR (test1) | live signed-in user: user1@cptdx.net | e1328847-0389-455c-a911-1f7bd1d1deee ###
=== Attempt 1: UPDATE the forceTunneling route (as the actor, raw ARM PUT) ===
----- RAW RESPONSE: UPDATE route (full ARM JSON, stdout + stderr) -----
ERROR: Forbidden({"error":{"code":"DenyAssignmentAuthorizationFailed","message":"The client 'user1@cptdx.net' with object id 'e1328847-0389-455c-a911-1f7bd1d1deee' has permission to perform action 'Microsoft.Network/routeTables/routes/write' on scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt/routes/forceTunneling'; however, the access is denied because of the deny assignment with name 'Deny modify and delete of protected route table (UDR)' and Id 'c703e3901de753a6874d4e268cbbbc85' at scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourcegroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt'."}})

----- deny 

### Attempt 2 - Add a new route (expected: BLOCKED)

**Expected output:** the same HTTP 403 deny response, this time for `Microsoft.Network/routeTables/routes/write` while creating `blockedRoute`. The new route is **not** created, and the classification line prints `>> CLASSIFICATION: BLOCKED by a deny assignment (expected).`

In [31]:
# CONTEXT: runs as the ACTOR (test1) - every az call uses AZURE_CONFIG_DIR=${TARGET_CFG}.
print_ctx "ACTOR (test1)" "${TARGET_CFG}"

echo "=== Attempt 2: ADD a new route (as the actor, raw ARM PUT) ==="
AZURE_CONFIG_DIR="${TARGET_CFG}" az rest --method put \
  --url "${RT_URL}/routes/blockedRoute?api-version=${ROUTES_API}" \
  --headers "Content-Type=application/json" \
  --body '{"properties":{"addressPrefix":"10.50.0.0/16","nextHopType":"VnetLocal"}}' \
  > /tmp/deny_create.txt 2>&1 || true

show_raw "CREATE route" /tmp/deny_create.txt
check_blocked /tmp/deny_create.txt

### RUNNING AS: ACTOR (test1) | live signed-in user: user1@cptdx.net | e1328847-0389-455c-a911-1f7bd1d1deee ###
=== Attempt 2: ADD a new route (as the actor, raw ARM PUT) ===
----- RAW RESPONSE: CREATE route (full ARM JSON, stdout + stderr) -----
ERROR: Forbidden({"error":{"code":"DenyAssignmentAuthorizationFailed","message":"The client 'user1@cptdx.net' with object id 'e1328847-0389-455c-a911-1f7bd1d1deee' has permission to perform action 'Microsoft.Network/routeTables/routes/write' on scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt/routes/blockedRoute'; however, the access is denied because of the deny assignment with name 'Deny modify and delete of protected route table (UDR)' and Id 'c703e3901de753a6874d4e268cbbbc85' at scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourcegroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt'."}})

----- deny assignment detail 

### Attempt 3 - Delete the route table (expected: BLOCKED)

**Expected output:** an HTTP 403 deny response for the `Microsoft.Network/routeTables/delete` action. The route table is **not** deleted, and the classification line prints `>> CLASSIFICATION: BLOCKED by a deny assignment (expected).`

In [32]:
# CONTEXT: runs as the ACTOR (test1) - every az call uses AZURE_CONFIG_DIR=${TARGET_CFG}.
print_ctx "ACTOR (test1)" "${TARGET_CFG}"

echo "=== Attempt 3: DELETE the route table (as the actor, raw ARM DELETE) ==="
AZURE_CONFIG_DIR="${TARGET_CFG}" az rest --method delete \
  --url "${RT_URL}?api-version=${ROUTES_API}" \
  > /tmp/deny_delete.txt 2>&1 || true

show_raw "DELETE route table" /tmp/deny_delete.txt
check_blocked /tmp/deny_delete.txt

### RUNNING AS: ACTOR (test1) | live signed-in user: user1@cptdx.net | e1328847-0389-455c-a911-1f7bd1d1deee ###
=== Attempt 3: DELETE the route table (as the actor, raw ARM DELETE) ===
----- RAW RESPONSE: DELETE route table (full ARM JSON, stdout + stderr) -----
ERROR: Forbidden({"error":{"code":"DenyAssignmentAuthorizationFailed","message":"The client 'user1@cptdx.net' with object id 'e1328847-0389-455c-a911-1f7bd1d1deee' has permission to perform action 'Microsoft.Network/routeTables/delete' on scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt'; however, the access is denied because of the deny assignment with name 'Deny modify and delete of protected route table (UDR)' and Id 'c703e3901de753a6874d4e268cbbbc85' at scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourcegroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt'."}})

----- deny assignment detail parsed fro

### Break-glass - Admin update (expected: SUCCEEDS)

**Expected output:** this cell runs in the **admin** context (the excluded principal from Step 3), so there is **no** deny error. The raw ARM PUT returns the updated route JSON (HTTP 200) with `nextHopIpAddress: 10.0.0.7`, and the final line prints `>> Admin (excluded) update SUCCEEDED - break-glass works while the actor stays blocked.`

In [33]:
# CONTEXT: runs as the ADMIN (ga1) - your default login, the EXCLUDED principal (no AZURE_CONFIG_DIR).
print_ctx "ADMIN (ga1)"

echo "=== Break-glass proof: the EXCLUDED admin (ga1) can STILL modify the UDR ==="
echo "(expected the excluded admin object id: ${MY_OBJECT_ID})"

az rest --method put \
  --url "${RT_URL}/routes/forceTunneling?api-version=${ROUTES_API}" \
  --headers "Content-Type=application/json" \
  --body '{"properties":{"addressPrefix":"0.0.0.0/0","nextHopType":"VirtualAppliance","nextHopIpAddress":"10.0.0.7"}}' \
  > /tmp/admin_update.txt 2>&1 || true

echo "----- RAW RESPONSE: admin UPDATE route (full ARM JSON) -----"
cat /tmp/admin_update.txt
echo "-----------------------------------------------------------"
if grep -qiE "deny assignment|disallowed|RequestDisallowedByAzure|AuthorizationFailed" /tmp/admin_update.txt; then
  echo ">> UNEXPECTED: the excluded admin was blocked - check the exclusion in Step 3."
else
  echo ">> Admin (excluded) update SUCCEEDED - break-glass works while the actor stays blocked."
fi

### RUNNING AS: ADMIN (ga1) | live signed-in user: ga1@cptdx.net | c1a139f7-6a1c-4c45-ae43-33c9b758e9c3 ###
=== Break-glass proof: the EXCLUDED admin (ga1) can STILL modify the UDR ===
(expected the excluded admin object id: c1a139f7-6a1c-4c45-ae43-33c9b758e9c3)
----- RAW RESPONSE: admin UPDATE route (full ARM JSON) -----
{
  "etag": "W/\"7a711109-0e4b-44f7-bdfe-7f3677455492\"",
  "id": "/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt/routes/forceTunneling",
  "name": "forceTunneling",
  "properties": {
    "addressPrefix": "0.0.0.0/0",
    "hasBgpOverride": false,
    "nextHopIpAddress": "10.0.0.7",
    "nextHopType": "VirtualAppliance",
    "provisioningState": "Succeeded"
  },
  "type": "Microsoft.Network/routeTables/routes"
}
-----------------------------------------------------------
>> Admin (excluded) update SUCCEEDED - break-glass works while the actor stays blocked.


## Observed result

| Operation | Without deny assignment | With deny assignment |
| --- | --- | --- |
| Update UDR route | Succeeds | **Blocked** |
| Add UDR route | Succeeds | **Blocked** |
| Delete route table | Succeeds | **Blocked** |

The deny assignment blocks the denied `write` / `delete` actions for **all principals**, overriding the Owner role. The error message returned by ARM names the specific deny assignment that caused the block.

## Step 5 - Audit effect (log without blocking)

The `2024-07-01-preview` deny-assignment schema exposes a `denyAssignmentEffect` property with two values: `enforced` (blocks access) and `audit` (logs the matching operations **without** blocking). In principle this lets you measure impact before switching to `enforced`.

> **Preview caveat:** `audit` is a preview capability and is **not yet honored in every tenant/region**. In this environment it is currently still **enforced** - reading the deny assignment back returns `effect: null` (the platform does not persist the value), and a non-excluded principal is still **blocked**. If you deploy `audit` here and leave the admin in scope, the admin's update fails with `DenyAssignmentAuthorizationFailed` - that is the preview limitation, not a bug in the demo.

> To keep the walkthrough moving, we exclude the **admin** (`MY_OBJECT_ID`) so the admin's update below succeeds and can be verified, and we still print the route-table write activity from the Activity Log. When `audit` becomes effective in your tenant, you would instead leave a principal **in scope** and watch its write succeed-but-logged.

> Activity Log events can take a few minutes to appear.


In [36]:
# CONTEXT: this cell runs as the ADMIN (your default login) - redeploy in audit mode + audited update.
print_ctx "ADMIN (ga1)"

# Azure still requires at least one excluded principal when targeting All Principals.
# NOTE: 'audit' is a preview effect and is NOT yet honored in every tenant/region.
# In this environment the deny is still ENFORCED (read-back shows effect: null), so we
# exclude the ADMIN here to ensure the admin's update below succeeds and can be verified.
# When audit becomes effective, you would instead leave a principal in scope to see its
# write succeed-but-logged.
cat > /tmp/audit.params.json <<EOF
{
  "\$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
  "contentVersion": "1.0.0.0",
  "parameters": {
    "routeTableName": { "value": "${ROUTE_TABLE}" },
    "denyAssignmentEffect": { "value": "audit" },
    "excludePrincipalIds": { "value": ["${MY_OBJECT_ID}"] }
  }
}
EOF

echo ">> Redeploying deny assignment with the 'audit' effect (admin excluded)..."
az deployment group create \
  --resource-group "${RG}" \
  --name denyassignment-audit \
  --template-file denyassignment.bicep \
  --parameters @/tmp/audit.params.json \
  -o table

echo ""
echo ">> Update the forceTunneling route as the excluded admin (expected to SUCCEED)..."
az network route-table route update \
  --resource-group "${RG}" \
  --route-table-name "${ROUTE_TABLE}" \
  --name forceTunneling \
  --next-hop-type VirtualAppliance \
  --next-hop-ip-address 10.0.0.8 \
  --query "{name:name, nextHopIp:nextHopIpAddress}" -o table

echo ""
echo ">> Recent route-table write activity (audit trail may take a few minutes to populate)..."
az monitor activity-log list \
  --resource-group "${RG}" \
  --offset 1h \
  --query "[?contains(authorization.action, 'routeTables')].{time:eventTimestamp, action:authorization.action, status:status.value, caller:caller}" \
  -o table

### RUNNING AS: ADMIN (ga1) | live signed-in user: ga1@cptdx.net | c1a139f7-6a1c-4c45-ae43-33c9b758e9c3 ###
>> Redeploying deny assignment with the 'audit' effect (admin excluded)...
Name                  State      Timestamp                         Mode         ResourceGroup
--------------------  ---------  --------------------------------  -----------  ---------------
denyassignment-audit  Succeeded  2026-06-23T12:47:10.433452+00:00  Incremental  cptdazdeny

>> Update the forceTunneling route as the excluded admin (expected to SUCCEED)...
Name            NextHopIp
--------------  -----------
forceTunneling  10.0.0.8

>> Recent route-table write activity (audit trail may take a few minutes to populate)...
Time                          Action                                      Status     Caller
----------------------------  ------------------------------------------  ---------  -------------
2026-06-23T12:39:48.3440991Z  Microsoft.Network/routeTables/routes/write  Succeeded  ga1@cptd

## Step 6 - Can a targeted principal delete the deny assignment to bypass it?

This step uses the **worst case on purpose**: the actor is a subscription **Owner**. The natural fix - also denying the deny assignment's own management action (`Microsoft.Authorization/denyAssignments/delete`) - is **not allowed** for user-assigned deny assignments. Azure rejects it at deploy time:

```
InvalidActionOrNotAction: 'Microsoft.Authorization/denyAssignments/delete' is not
permitted in user assigned deny assignments. Denying delete access to deny
assignments is not allowed.
```

(The same applies to `denyAssignments/write`.) So a user-assigned deny assignment cannot make *itself* tamper-proof.

### So what is the deny assignment actually worth? (You do NOT need a custom role)

The bypass is only possible for a principal that holds `Microsoft.Authorization/denyAssignments/delete`. Among the built-in roles, **only three** have it: **Owner**, **User Access Administrator**, and **Role Based Access Control Administrator** (they carry `Microsoft.Authorization/*`). **Every other built-in role cannot delete a deny assignment** - for example **Contributor** and **Network Contributor** explicitly list `Microsoft.Authorization/*/Delete` and `/Write` in their `NotActions`.

That is exactly where the deny assignment adds value that plain RBAC cannot:

| Principal's role | Can modify/delete the route table? | Can delete the deny assignment? |
| --- | --- | --- |
| **Contributor / Network Contributor** | No - **blocked by the deny assignment** | **No** - lacks `denyAssignments/delete` |
| **Owner / User Access Admin / RBAC Admin** | No - blocked by the deny assignment | **Yes** - can remove it, then change the route |

So for the **common case** (Contributors and resource-specific roles) the deny assignment is fully effective **with no custom role at all** - it blocks the one thing RBAC roles cannot express (a *deny* on write/delete), and those roles already cannot touch deny assignments. You only have a residual bypass for the **three highly privileged authorization-managing roles**, and the right mitigation there is plain least-privilege:

- **Do not grant Owner / User Access Administrator / Role Based Access Control Administrator** to the people you are trying to restrict. Give them Contributor (or a narrower resource role) instead - then there is no bypass and no custom role is needed.
- If you must restrict even an Owner, use a **deployment stack** with `denySettings.mode = denyDelete`; Azure then creates a **system-protected** deny assignment (`isSystemProtected = true`) that nobody, not even an Owner, can delete.

The cell below proves the platform restriction (the `denyAssignments/delete` deploy is rejected) and reads `isSystemProtected` back (`false`). Step 6b then shows the Owner bypass in action and restores protection.


In [37]:
# CONTEXT: this cell runs as the ADMIN (your default login).
print_ctx "ADMIN (ga1)"

# 1) Redeploy a clean deny assignment (enforced, admin excluded) so the demo state
#    is consistent and the excluded admin keeps control.
echo ">> Redeploying the deny assignment (enforced, admin excluded)..."
cat > /tmp/scoped.params.json <<EOF
{
  "\$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
  "contentVersion": "1.0.0.0",
  "parameters": {
    "routeTableName": { "value": "${ROUTE_TABLE}" },
    "excludePrincipalIds": { "value": ["${MY_OBJECT_ID}"] }
  }
}
EOF
az deployment group create \
  --resource-group "${RG}" \
  --name denyassignment-scoped \
  --template-file denyassignment.bicep \
  --parameters @/tmp/scoped.params.json \
  -o table

export DENY_ID=$(az deployment group show -g "${RG}" -n denyassignment-scoped --query properties.outputs.denyAssignmentId.value -o tsv)
echo "Deny assignment resource ID: ${DENY_ID}"

# 2) Prove that you CANNOT make it self-protecting: try to create a deny assignment
#    that denies 'denyAssignments/delete'. Azure rejects this for user-assigned deny
#    assignments with InvalidActionOrNotAction.
echo ""
echo ">> Attempting to deny 'denyAssignments/delete' (expected to be REJECTED by Azure)..."
SELF_URL="https://management.azure.com/subscriptions/${SUBSCRIPTION_ID}/resourceGroups/${RG}/providers/Microsoft.Network/routeTables/${ROUTE_TABLE}/providers/Microsoft.Authorization/denyAssignments/$(cat /proc/sys/kernel/random/uuid)?api-version=2024-07-01-preview"
az rest --method put \
  --url "${SELF_URL}" \
  --headers "Content-Type=application/json" \
  --body "{\"properties\":{\"denyAssignmentName\":\"selfprotect-test\",\"permissions\":[{\"actions\":[\"Microsoft.Authorization/denyAssignments/delete\"]}],\"principals\":[{\"id\":\"${TARGET_USER_OBJECT_ID}\",\"type\":\"User\"}]}}" \
  > /tmp/selfprotect.txt 2>&1 || true
cat /tmp/selfprotect.txt
grep -qiE "InvalidActionOrNotAction|not permitted in user assigned deny assignments" /tmp/selfprotect.txt \
  && echo ">> CONFIRMED: Azure forbids denying 'denyAssignments/delete' - no self-protection for user-assigned deny assignments." \
  || echo ">> UNEXPECTED: the self-protection action was not rejected - check the response above."

# 3) The live deny assignment is therefore user-owned and deletable (isSystemProtected = false).
echo ""
echo ">> Reading isSystemProtected of the live deny assignment..."
az rest --method get \
  --url "https://management.azure.com${DENY_ID}?api-version=2024-07-01-preview" \
  --query "{displayName:properties.denyAssignmentName, isSystemProtected:properties.isSystemProtected, actions:properties.permissions[0].actions}" -o jsonc

echo ""
echo "Because isSystemProtected = false, the targeted Owner (actor) can delete this"
echo "deny assignment. The next cell runs as that actor and demonstrates the bypass."

### RUNNING AS: ADMIN (ga1) | live signed-in user: ga1@cptdx.net | c1a139f7-6a1c-4c45-ae43-33c9b758e9c3 ###
>> Redeploying the deny assignment (enforced, admin excluded)...
Name                   State      Timestamp                         Mode         ResourceGroup
---------------------  ---------  --------------------------------  -----------  ---------------
denyassignment-scoped  Succeeded  2026-06-23T12:54:46.688571+00:00  Incremental  cptdazdeny
Deny assignment resource ID: /subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt/providers/Microsoft.Authorization/denyAssignments/c703e390-1de7-53a6-874d-4e268cbbbc85

>> Attempting to deny 'denyAssignments/delete' (expected to be REJECTED by Azure)...
ERROR: Bad Request({"error":{"code":"InvalidActionOrNotAction","message":"'Microsoft.Authorization/denyAssignments/delete' is not permitted in user assigned deny assignments. Denying delete access to deny assi

### Step 6b - The Owner bypass in action

Every command below runs with `AZURE_CONFIG_DIR=${TARGET_CFG}`, i.e. **as `e1328847-0389-455c-a911-1f7bd1d1deee`**, not as the admin. The actor is an `Owner` but is covered by the deny assignment, so:

1. updating the route → **blocked** by the deny assignment (the deny still works for resource actions);
2. deleting the deny assignment → **succeeds**, because a user-assigned deny assignment is not system protected and cannot deny its own `delete`. This is the bypass that **cannot** be closed for user-assigned deny assignments.

The final block switches back to the **admin** and **redeploys** the deny assignment to restore protection (and to leave the demo in a clean, protected state for the cleanup step).


In [39]:
# CONTEXT: runs as the ACTOR (test1) - every az call uses AZURE_CONFIG_DIR=${TARGET_CFG}.
print_ctx "ACTOR (test1)" "${TARGET_CFG}"
echo "(expected the actor object id: ${TARGET_USER_OBJECT_ID})"

echo ""
echo "=== Attempt 1: UPDATE the route (expected: BLOCKED) ==="
AZURE_CONFIG_DIR="${TARGET_CFG}" az network route-table route update \
  --resource-group "${RG}" \
  --route-table-name "${ROUTE_TABLE}" \
  --name forceTunneling \
  --next-hop-type VirtualAppliance \
  --next-hop-ip-address 10.0.0.11 \
  > /tmp/t_update.txt 2>&1 || true
cat /tmp/t_update.txt
grep -qiE "deny assignment|denyassignment|disallowed|RBACAccessDenied|AuthorizationFailed" /tmp/t_update.txt \
  && echo ">> Route update BLOCKED by the deny assignment (expected)." \
  || echo ">> Route update was NOT blocked - check role-assignment propagation."

echo ""
echo "=== Attempt 2: DELETE the deny assignment - the bypass (expected: SUCCEEDS as Owner) ==="
# NOTE on classification: a SUCCESSFUL delete returns the deleted deny assignment's
# JSON body (which naturally contains the word "denyAssignment"), so we must NOT
# classify on that word. Only a genuine authorization failure (ERROR: / Forbidden /
# AuthorizationFailed / disallowed) means the delete was blocked.
AZURE_CONFIG_DIR="${TARGET_CFG}" az rest --method delete \
  --url "https://management.azure.com${DENY_ID}?api-version=2024-07-01-preview" \
  > /tmp/t_deny.txt 2>&1 || true
cat /tmp/t_deny.txt
if grep -qiE "^ERROR:|Forbidden|AuthorizationFailed|disallowed|RBACAccessDenied|does not have authorization" /tmp/t_deny.txt; then
  echo ">> Deny-assignment delete was BLOCKED - the actor lacks denyAssignments/delete (e.g. a Contributor would land here)."
else
  echo ">> Deny-assignment delete SUCCEEDED - this Owner bypassed the protection. Only Owner / User Access Admin / RBAC Admin can; a Contributor could NOT."
fi

echo ""
echo "=== Restore: ADMIN redeploys the deny assignment to re-protect the route table ==="
# CONTEXT switches back to the ADMIN (no AZURE_CONFIG_DIR) to put the protection back.
print_ctx "ADMIN (ga1)"
az deployment group create \
  --resource-group "${RG}" \
  --name denyassignment-restore \
  --template-file denyassignment.bicep \
  --parameters @/tmp/scoped.params.json \
  -o table
export DENY_ID=$(az deployment group show -g "${RG}" -n denyassignment-restore --query properties.outputs.denyAssignmentId.value -o tsv)
echo "Deny assignment restored. Resource ID: ${DENY_ID}"

### RUNNING AS: ACTOR (test1) | live signed-in user: user1@cptdx.net | e1328847-0389-455c-a911-1f7bd1d1deee ###
(expected the actor object id: e1328847-0389-455c-a911-1f7bd1d1deee)

=== Attempt 1: UPDATE the route (expected: BLOCKED) ===
ERROR: (DenyAssignmentAuthorizationFailed) The client 'user1@cptdx.net' with object id 'e1328847-0389-455c-a911-1f7bd1d1deee' has permission to perform action 'Microsoft.Network/routeTables/routes/write' on scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourceGroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt/routes/forceTunneling'; however, the access is denied because of the deny assignment with name 'Deny modify and delete of protected route table (UDR)' and Id 'c703e3901de753a6874d4e268cbbbc85' at scope '/subscriptions/7777febc-d5d8-44da-b990-643c8369f1e1/resourcegroups/cptdazdeny/providers/Microsoft.Network/routeTables/cptdazdeny-rt'.
Code: DenyAssignmentAuthorizationFailed
Message: The client 'user1@cptdx.net' w

## Cleanup

**Order matters**: because the deny assignment blocks `routeTables/delete`, deleting the resource group would also be blocked while the deny is in place. Remove the deny assignment first, then delete the resource group. You can remove the deny assignment because either self-protection is off (Steps 3-5) or you are an excluded principal (Step 6). The target user's Owner grant is removed too (and would also be cleaned up by the resource-group deletion).

In [ ]:
# CONTEXT: this cell runs as the ADMIN (your default login) - the excluded principal that can clean up.
print_ctx "ADMIN (ga1)"

echo ">> Removing the deny assignment..."
az rest --method delete \
  --url "https://management.azure.com${DENY_ID}?api-version=2024-07-01-preview"
echo "Deny assignment removed."

if [ -n "${TARGET_USER_OBJECT_ID:-}" ]; then
  echo ">> Removing the actor's Owner role assignment on the subscription..."
  az role assignment delete \
    --assignee "${TARGET_USER_OBJECT_ID}" \
    --role "Owner" \
    --scope "/subscriptions/${SUBSCRIPTION_ID}" 2>/dev/null || true
  echo "Role assignment removed (if it existed)."
fi

if [ -n "${TARGET_CFG:-}" ] && [ -d "${TARGET_CFG}" ]; then
  echo ">> Signing out and removing the target-user CLI context..."
  AZURE_CONFIG_DIR="${TARGET_CFG}" az logout 2>/dev/null || true
  rm -rf "${TARGET_CFG}"
  echo "Target-user context removed."
fi

echo ">> Deleting the resource group (async)..."
az group delete --name "${RG}" --yes --no-wait
echo "Cleanup initiated for resource group ${RG}."

## Summary & key takeaways

- **Deny assignments override all role assignments**, including `Owner`. They are the right tool when you must guarantee that a resource (like a force-tunneling UDR) cannot be changed or removed - this is the one thing plain RBAC roles cannot express (there is no "deny" role).
- User-assigned deny assignments are declarable in **Bicep** via `Microsoft.Authorization/denyAssignments@2024-07-01-preview` and deployable with the **Azure CLI** (`az deployment group create`).
- When you target **All Principals**, Azure **requires at least one excluded principal** - so a **break-glass** admin exclusion is built in from the start (Step 3). The excluded admin keeps full control while every other principal is blocked.
- Scope the deny **narrowly** (here: the single route table) to limit blast radius.
- Use the `audit` effect (Step 5) to **measure impact** before switching to `enforced`. Note that `audit` is a **preview** capability and may still **enforce** in some tenants.
- A **user-assigned** deny assignment **cannot protect itself** from deletion (Azure rejects denying `denyAssignments/delete` with `InvalidActionOrNotAction`). But this only matters for the **three** roles that can delete deny assignments - **Owner**, **User Access Administrator**, **Role Based Access Control Administrator**. **No custom role is needed**: a **Contributor** (or narrower role) is already blocked from both changing the route table *and* deleting the deny assignment (Step 6). To restrict even an Owner, use a **system-protected** deny assignment via a deployment stack (`denySettings.mode = denyDelete`).
- Remember the **ARM authorization cache** (up to 30 minutes) and to **remove the deny assignment before deleting** protected resources.
- For a comparison with the Azure Policy `denyAction` approach in `denyudrdelete/`, see [README.md](README.md).

### References

- [Azure deny assignments](https://learn.microsoft.com/azure/role-based-access-control/deny-assignments)
- [New-AzDenyAssignment](https://learn.microsoft.com/powershell/module/az.resources/new-azdenyassignment)
- [Microsoft.Authorization/denyAssignments (Bicep)](https://learn.microsoft.com/azure/templates/microsoft.authorization/denyassignments)
